In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

In [30]:
# Convert MNIST image files into a tensor of 4D (N of img, height, weight, color channel)
transform = transforms.ToTensor()

In [31]:
# Train data
train_data = datasets.MNIST(root='cnn_data', train=True, download=True, transform=transform)


In [32]:
# Test data
test_data = datasets.MNIST(root='cnn_data', train=False, download=True, transform=transform)


In [33]:
# Creating a small batch size for image
train_load = DataLoader(train_data, batch_size=10, shuffle=True)
test_load = DataLoader(test_data, batch_size=10, shuffle=False)   

In [34]:
# Define Our CNN Model
# Describe convolutional layer and what it's doing (2 convolutional layers)
# This is just an example 
conv1 = nn.Conv2d(1, 6, 3, 1)       # 1 input, 6 output, 3x3 kernel, 1 stride(how many pixels will moved for filtering)
conv2 = nn.Conv2d(6, 16, 3, 1)      

In [35]:
# Grab 1 MNIST record/image
for i, (X_Train, y_train) in enumerate(train_data):
    break

# Explanation
i = 0

X_Train = image_0

y_train = label_0


### break
Immediately stops the loop after the first iteration.
So you only get one image and its label, not the entire dataset.

In [36]:
X_Train.shape

torch.Size([1, 28, 28])

In [37]:
# Reshapes the MNIST image into a 4D tensor so it can be passed into a CNN
x = X_Train.view(1, 1, 28, 28)      # 1 batch, 1 image, 28 by 28

## Why do we need this?
Because nn.Conv2d expects a 4D tensor when processing a batch of images.

conv1 = nn.Conv2d(1, 6, kernel_size=3, stride=1)

output = conv1(x)

print(output.shape)
 
torch.Size([1, 6, 26, 26])

In [38]:
# Perform our first convolution
x = F.relu(conv1(x)) # Rectified Linear Unit for our activation function

1. conv1(x) → Applies a convolution to the image.

2. F.relu(...) → Applies the ReLU activation function.

It replaces negative values with zero while keeping positive values unchanged.

## Why do we need ReLU?
Without activation functions, stacking convolutional layers would only produce a linear transformation (before considering other operations).

ReLU introduces non-linearity, allowing the CNN to learn more complex patterns.

In [39]:
x.shape     # 1 image, 6 filter (what we asked for), 26 by 26 (we have padding, and useless pixels from our image will just drop)

torch.Size([1, 6, 26, 26])

In [40]:
# pass thru the pulling layer 
x = F.max_pool2d(x,2,2)     # our x by 2 kernal, 2 stride
x.shape # 1 image, 6 filter, 13 by 13 (26 / 2 = 13)

torch.Size([1, 6, 13, 13])

We use here max pulling, it takes form filter layer our data and do argmax(data) = if we have 1, 3, 4, 2
it will return 4

In [41]:
# Do our second convolutional layer
x = F.relu(conv2(x))
x.shape     # Here we again lose our 2 pixels, for dont using padding

torch.Size([1, 16, 11, 11])

In [42]:
# Pooling layer
x = F.max_pool2d(x,2,2)
x.shape

torch.Size([1, 16, 5, 5])

In [44]:
# Model Class
class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()      # подготовка базового функционала PyTorch.
        self.conv1 = nn.Conv2d(1, 6, 3, 1)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        # Fully Connected
        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
        # We are just creating whole blocks, we willdnt give the numbers for calculating
    
    def forward(self, X):
        # First block
        X = self.conv1(X)
        X = F.relu(X)
        X = F.max_pool2d(X,2,2)
        
        # Second block
        X = self.conv2(X)
        X = F.relu(X)
        X = F.max_pool2d(X,2,2)
        
        # Re-View to flatten it out
        X = X.view(-1, 16*5*5) # negative one so that we can use any batch size
        
        # Fully Connected Layers
        X = self.fc1(X)
        X = F.relu(X)
        X = self.fc2(X)
        X = F.relu(X)
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)
    
        # Here we are giving the frequence for our blocks what will be after what 
        


In [45]:
# Create an instance of pur model
torch.manual_seed(41)
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [ ]:
# Loss Func Optimizer
criterion = nn.CrossEntropyLoss()
